In [31]:
!pip install ultralytics gradio

In [32]:
import os

# Create the weights directory as required by the app
os.makedirs('weights', exist_ok=True)

# Extract and write pipeline code from cell ymNfbyOkja2Q to a file
pipeline_code = """
import os
import shutil
import subprocess
import tempfile
from collections import Counter, defaultdict
from dataclasses import dataclass

import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO

CFG = {
    "det": os.getenv("DET_WEIGHTS", "weights/detector_seg.pt"),
    "cls": os.getenv("CLS_WEIGHTS", "weights/species_cls.pt"),
}
DET_IMGSZ = 640
CLS_IMGSZ = 224
MAX_VOTES_PER_TRACK = 12
SIZE_LABELS = ["Small", "Medium", "Large"]

_models = {"det": None, "cls": None}

@dataclass
class Settings:
    conf: float = 0.25
    iou: float = 0.5
    t_small: float = 0.08
    t_large: float = 0.20
    do_species: bool = True
    classify_every: int = 5
    max_frames: int = 900
    px_per_cm: float = 0.0
    trail_len: int = 40
    min_track_frames: int = 3

def set_weights(det=None, cls=None):
    if det:
        CFG["det"] = det
    if cls:
        CFG["cls"] = cls
    reload_models()

def reload_models():
    _models["det"] = None
    _models["cls"] = None

def _load(path, fallback=None):
    if path and os.path.exists(path):
        return YOLO(path)
    if fallback:
        print(f"[warn] {path} not found, using {fallback}")
        return YOLO(fallback)
    print(f"[warn] {path} not found")
    return None

def get_detector(fresh=False, weights=None):
    if weights:
        return YOLO(weights)
    if fresh:
        return _load(CFG["det"], "yolo11s-seg.pt")
    if _models["det"] is None:
        _models["det"] = _load(CFG["det"], "yolo11s-seg.pt")
    return _models["det"]

def get_classifier():
    if _models["cls"] is None:
        _models["cls"] = _load(CFG["cls"])
    return _models["cls"]

def model_status():
    d = "found" if os.path.exists(CFG["det"]) else "MISSING (using stock COCO yolo11s-seg)"
    c = "found" if os.path.exists(CFG["cls"]) else "MISSING (species = Unknown)"
    return f"Detector `{CFG['det']}`: {d}  \\nClassifier `{CFG['cls']}`: {c}"

def classify_crop(crop_bgr):
    clf = get_classifier()
    if clf is None or crop_bgr is None or crop_bgr.size == 0:
        return "Unknown", 0.0
    res = clf.predict(crop_bgr, imgsz=CLS_IMGSZ, verbose=False)[0]
    idx = int(res.probs.top1)
    return clf.names[idx], float(res.probs.top1conf)

def mask_pixels(poly, shape):
    h, w = shape[:2]
    x0, y0 = np.floor(poly.min(0)).astype(int)
    x1, y1 = np.ceil(poly.max(0)).astype(int)
    x0, y0 = max(x0, 0), max(y0, 0)
    x1, y1 = min(x1, w - 1), min(y1, h - 1)
    if x1 <= x0 or y1 <= y0:
        return None
    canvas = np.zeros((y1 - y0 + 1, x1 - x0 + 1), np.uint8)
    cv2.fillPoly(canvas, [(poly - np.array([x0, y0])).astype(np.int32)], 1)
    ys, xs = np.nonzero(canvas)
    if len(xs) < 5:
        return None
    return np.stack([xs + x0, ys + y0], axis=1).astype(np.float32)

def pca_length(points):
    pts = points - points.mean(0)
    _, _, vt = np.linalg.svd(pts, full_matrices=False)
    proj = pts @ vt[0]
    return float(proj.max() - proj.min() + 1.0)

def size_class(rel_len, t_small, t_large):
    if rel_len < t_small:
        return SIZE_LABELS[0]
    if rel_len < t_large:
        return SIZE_LABELS[1]
    return SIZE_LABELS[2]

def colour_for(key):
    rng = np.random.RandomState(abs(hash(str(key))) % (2**31))
    return tuple(int(c) for c in rng.randint(60, 255, 3))

def draw_detection(img, det, alpha=0.35):
    colour = colour_for(det["id"] if det["id"] is not None else det["species"])
    poly = det.get("poly")
    if poly is not None and len(poly) >= 3:
        overlay = img.copy()
        cv2.fillPoly(overlay, [poly.astype(np.int32)], colour)
        cv2.addWeighted(overlay, alpha, img, 1 - alpha, 0, img)
        cv2.polylines(img, [poly.astype(np.int32)], True, colour, 2)
    x1, y1, x2, y2 = [int(v) for v in det["box"]]
    cv2.rectangle(img, (x1, y1), (x2, y2), colour, 2)
    tag = f"ID {det['id']} | " if det["id"] is not None else ""
    label = f"{tag}{det['species']} {det['sconf']:.0%} | {det['size']}"
    if det.get("length_cm") is not None:
        label += f" | {det['length_cm']:.1f} cm"
    (tw, th), bl = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
    ty = max(y1, th + 6)
    cv2.rectangle(img, (x1, ty - th - 6), (x1 + tw + 4, ty + bl - 4), colour, -1)
    cv2.putText(img, label, (x1 + 2, ty - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)

def extract_detections(result, frame, st, votes=None, frame_idx=0, scale=1.0):
    dets = []
    if result.boxes is None or len(result.boxes) == 0:
        return dets
    h, w = frame.shape[:2]
    diag = float(np.hypot(h, w))
    boxes = result.boxes.xyxy.cpu().numpy()
    confs = result.boxes.conf.cpu().numpy()
    ids = (result.boxes.id.cpu().numpy().astype(int) if result.boxes.id is not None else [None] * len(boxes))
    polys = result.masks.xy if result.masks is not None else [None] * len(boxes)
    for box, conf, tid, poly in zip(boxes, confs, ids, polys):
        x1, y1 = max(int(box[0]), 0), max(int(box[1]), 0)
        x2, y2 = min(int(box[2]), w), min(int(box[3]), h)
        if x2 - x1 < 4 or y2 - y1 < 4:
            continue
        pts = None
        if poly is not None and len(poly) >= 3:
            pts = mask_pixels(np.asarray(poly), frame.shape)
        if pts is None:
            pts = np.array([[x1, y1], [x2, y1], [x2, y2], [x1, y2]], np.float32)
        length_px = pca_length(pts)
        rel = length_px / diag
        length_cm = None
        if st.px_per_cm and st.px_per_cm > 0:
            length_cm = (length_px / scale) / st.px_per_cm
        species, sconf = "Unknown", 0.0
        if st.do_species:
            crop = frame[y1:y2, x1:x2]
            if tid is None or votes is None:
                species, sconf = classify_crop(crop)
            else:
                v = votes[tid]
                n = len(v["confs"])
                if n == 0 or (n < MAX_VOTES_PER_TRACK and frame_idx % max(st.classify_every, 1) == 0):
                    sp, sc = classify_crop(crop)
                    v["votes"][sp] += sc
                    v["confs"].append(sc)
                species = v["votes"].most_common(1)[0][0]
                sconf = float(np.mean(v["confs"]))
        dets.append(dict(
            id=tid, box=(x1, y1, x2, y2), poly=None if poly is None else np.asarray(poly),
            conf=float(conf), species=species, sconf=sconf, length_px=length_px,
            rel=rel, length_cm=length_cm, size=size_class(rel, st.t_small, st.t_large),
            centre=((x1 + x2) / 2.0, (y1 + y2) / 2.0),
        ))
    return dets

def export_csv(df, name):
    path = os.path.join(tempfile.mkdtemp(), name)
    df.to_csv(path, index=False)
    return path

IMG_COLS = ["Fish", "Species", "Species conf.", "Size class", "Rel. length (% diag)", "Length (cm)", "Det. conf."]

def process_image(image_rgb, st, det_weights=None):
    det_model = get_detector(weights=det_weights)
    bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)
    res = det_model.predict(bgr, imgsz=DET_IMGSZ, conf=st.conf, iou=st.iou, verbose=False)[0]
    dets = extract_detections(res, bgr, st)
    out = bgr.copy()
    for d in dets:
        draw_detection(out, d)
    rows = [{
        "Fish": i + 1, "Species": d["species"], "Species conf.": round(d["sconf"], 3),
        "Size class": d["size"], "Rel. length (% diag)": round(d["rel"] * 100, 1),
        "Length (cm)": None if d["length_cm"] is None else round(d["length_cm"], 1),
        "Det. conf.": round(d["conf"], 3),
    } for i, d in enumerate(dets)]
    df = pd.DataFrame(rows, columns=IMG_COLS)
    return cv2.cvtColor(out, cv2.COLOR_BGR2RGB), dets, df

def _reencode_h264(src):
    if not shutil.which("ffmpeg"):
        return src
    dst = src.replace(".mp4", "_h264.mp4")
    try:
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", src,
                        "-vcodec", "libx264", "-pix_fmt", "yuv420p", dst], check=True)
        return dst
    except Exception:
        return src

def _movement(path, fps, first, last, diag):
    p = np.asarray(path, np.float32)
    if len(p) < 2:
        return 0.0, 0.0, 0.0, 0.0
    steps = np.linalg.norm(np.diff(p, axis=0), axis=1)
    path_len = float(steps.sum())
    net = p[-1] - p[0]
    disp = float(np.linalg.norm(net))
    dur = max((last - first) / fps, 1e-6)
    heading = float(np.degrees(np.arctan2(-net[1], net[0])))
    return path_len / diag * 100, disp / diag * 100, path_len / diag * 100 / dur, heading

def process_video(video_path, st, write_video=True, max_width=1280, progress=None, det_weights=None):
    det_model = get_detector(fresh=True, weights=det_weights)
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError("Could not open the video.")
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 0
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    scale = min(1.0, max_width / W) if (write_video and max_width) else 1.0
    outW, outH = max(int(W * scale) // 2 * 2, 2), max(int(H * scale) // 2 * 2, 2)
    diag = float(np.hypot(outW, outH))
    writer, raw_path = None, None
    if write_video:
        raw_path = os.path.join(tempfile.mkdtemp(), "annotated.mp4")
        writer = cv2.VideoWriter(raw_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (outW, outH))
    votes = defaultdict(lambda: {"votes": Counter(), "confs": []})
    info = defaultdict(lambda: {"rel": [], "px": [], "det": [], "first": None, "last": None, "n": 0, "path": []})
    records, frame_rows = [], []
    limit = int(st.max_frames) if st.max_frames else (total or 10**9)
    n_total = min(total, limit) if total else 0
    idx = 0
    while idx < limit:
        ok, frame = cap.read()
        if not ok:
            break
        if scale < 1.0:
            frame = cv2.resize(frame, (outW, outH))
        res = det_model.track(frame, persist=True, tracker="bytetrack.yaml", imgsz=DET_IMGSZ, conf=st.conf, iou=st.iou, verbose=False)[0]
        dets = extract_detections(res, frame, st, votes=votes, frame_idx=idx, scale=scale)
        for d in dets:
            tid = d["id"]
            if tid is None:
                continue
            ti = info[tid]
            ti["rel"].append(d["rel"])
            ti["px"].append(d["length_px"] / scale)
            ti["det"].append(d["conf"])
            ti["first" ] = idx if ti["first"] is None else ti["first"]
            ti["last"] = idx
            ti["n"] += 1
            ti["path"].append(d["centre"])
            d["size"] = size_class(float(np.median(ti["rel"])), st.t_small, st.t_large)
            x1, y1, x2, y2 = d["box"]
            records.append((idx, tid, x1 / scale, y1 / scale, x2 / scale, y2 / scale))
            frame_rows.append({
                "frame": idx + 1, "time_s": round(idx / fps, 3), "track_id": tid,
                "species": d["species"], "species_conf": round(d["sconf"], 3),
                "size_class": d["size"], "rel_length_pct_diag": round(d["rel"] * 100, 2),
                "length_cm": None if d["length_cm"] is None else round(d["length_cm"], 2),
                "x1": round(x1 / scale, 1), "y1": round(y1 / scale, 1),
                "x2": round(x2 / scale, 1), "y2": round(y2 / scale, 1),
            })
        if write_video:
            for d in dets:
                if d["id"] is None:
                    continue
                path = info[d["id"]]["path"][-max(int(st.trail_len), 2):]
                if len(path) >= 2:
                    cv2.polylines(frame, [np.array(path, np.int32)], False, colour_for(d["id"]), 2, cv2.LINE_AA)
            for d in dets:
                draw_detection(frame, d)
            seen = sum(1 for t in info.values() if t["n"] >= st.min_track_frames)
            cv2.rectangle(frame, (0, 0), (270, 30), (0, 0, 0), -1)
            cv2.putText(frame, f"Unique fish so far: {seen}", (6, 21), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1, cv2.LINE_AA)
            writer.write(frame)
        idx += 1
    cap.release()
    if writer is not None:
        writer.release()
    out_video = _reencode_h264(raw_path) if write_video else None
    rows = []
    for tid, ti in sorted(info.items()):
        if ti["n"] < st.min_track_frames:
            continue
        v = votes[tid]
        species = v["votes"].most_common(1)[0][0] if v["votes"] else "Unknown"
        sconf = float(np.mean(v["confs"])) if v["confs"] else 0.0
        rel = float(np.median(ti["rel"]))
        path_len, disp, speed, heading = _movement(ti["path"], fps, ti["first"], ti["last"], diag)
        row = {
            "Track ID": tid, "Species": species, "Species conf.": round(sconf, 3),
            "Size class": size_class(rel, st.t_small, st.t_large),
            "Rel. length (% diag)": round(rel * 100, 1),
            "Length (cm)": (round(float(np.median(ti["px"])) / st.px_per_cm, 1) if st.px_per_cm and st.px_per_cm > 0 else None),
            "Frames seen": ti["n"],
            "First (s)": round(ti["first"] / fps, 1), "Last (s)": round(ti["last"] / fps, 1),
            "Path (% diag)": round(path_len, 1), "Net disp. (% diag)": round(disp, 1),
            "Speed (% diag/s)": round(speed, 1), "Heading (deg)": round(heading, 0),
        }
        rows.append(row)
    return dict(out_video=out_video, tracks=pd.DataFrame(rows), frames=pd.DataFrame(frame_rows), frame_records=records, fps=fps, n_frames=idx, count=len(rows))

def summarize_tracks(tracks, n_frames, fps, calibrated):
    s = f"### Unique fish (de-duplicated by ByteTrack): **{len(tracks)}**\\n"
    s += f"_Processed {n_frames} frames at {fps:.1f} FPS._\\n\\n"
    if len(tracks):
        sp = Counter(tracks["Species"])
        s += "\\n".join(f"- {k}: {v}" for k, v in sp.most_common())
        sz = Counter(tracks["Size class"])
        s += "\\n\\n**Size classes:** " + ", ".join(f"{k}: {v}" for k, v in sz.items())
        s += f"\\n\\n**Mean speed:** {tracks['Speed (% diag/s)'].mean():.1f} % of frame diagonal / s"
    return s
"""
with open('pipeline.py', 'w') as f:
    f.write(pipeline_code)

# Extract and write dataset_check code from cell I6uowKFdjX8M to a file
dataset_check_code = """
import random
import re
from collections import Counter
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import pipeline

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
VID_EXT = {".mp4", ".avi", ".mov", ".mkv"}
NAMES = ["DeepFish", "OZFish", "Brackish", "Fish4Knowledge"]
SPECIES_SET = "Fish4Knowledge"
DEFAULT_PATHS = {
    "DeepFish": "/content/data/deepfish",
    "OZFish": "/content/data/ozfish",
    "Brackish": "/content/data/brackish",
    "Fish4Knowledge": "/content/data/fish4knowledge",
}
MAX_GALLERY = 24
_cache = {}

def scan(path, refresh=False):
    p = Path(str(path).strip())
    if not p.exists() or not p.is_dir():
        raise ValueError(f"Folder not found: {path}")
    key = str(p.resolve())
    if refresh or key not in _cache:
        imgs, vids = [], []
        for f in sorted(p.rglob("*")):
            ext = f.suffix.lower()
            if ext in IMG_EXT:
                imgs.append(f)
            elif ext in VID_EXT:
                vids.append(f)
        _cache[key] = (p, imgs, vids)
    return _cache[key]

def scan_summary(path, refresh=False):
    root, imgs, vids = scan(path, refresh)
    sub = Counter()
    for f in imgs:
        rel = f.relative_to(root)
        sub[rel.parts[0] if len(rel.parts) > 1 else "(top level)"] += 1
    return {
        "n_images": len(imgs), "n_videos": len(vids),
        "videos": [str(v.relative_to(root)) for v in vids[:500]],
        "subfolders": dict(sub.most_common(30)),
        "n_subfolders": len(sub),
    }

def resolve_video(path, rel):
    root, _, _ = scan(path)
    f = (root / rel)
    if not f.exists():
        raise ValueError(f"Video not found: {f}")
    return str(f)

def label_count(img_path):
    parts = list(Path(img_path).parts)
    lp = None
    for i in range(len(parts) - 1, -1, -1):
        if parts[i] == "images":
            parts[i] = "labels"
            lp = Path(*parts).with_suffix(".txt")
            break
    if lp is None:
        lp = Path(img_path).with_suffix(".txt")
    if not lp.exists():
        return None
    return sum(1 for ln in lp.read_text().splitlines() if ln.strip())

def _thumb(rgb, side=640):
    h, w = rgb.shape[:2]
    s = side / max(h, w)
    return cv2.resize(rgb, (int(w * s), int(h * s))) if s < 1 else rgb

def _norm(s):
    return re.sub(r"[\\W_]+", "", str(s).lower())

def check_species(imgs, per_class, seed):
    clf = pipeline.get_classifier()
    if clf is None:
        raise ValueError("Species classifier weights not found - set them in 'Models & settings'.")
    by = {}
    for f in imgs:
        by.setdefault(f.parent.name, []).append(f)
    if len(by) < 2:
        raise ValueError("Expected one sub-folder per species (found fewer than 2 folders).")
    rng = random.Random(seed)
    model_names = {_norm(v) for v in clf.names.values()}
    rows, gallery = [], []
    for cls, files in sorted(by.items()):
        for f in rng.sample(files, min(int(per_class), len(files))):
            im = cv2.imread(str(f))
            if im is None:
                continue
            sp, sc = pipeline.classify_crop(im)
            rows.append({"file": f.name, "true species": cls, "predicted": sp, "confidence": round(sc, 3)})
            if len(gallery) < MAX_GALLERY:
                gallery.append((_thumb(cv2.cvtColor(im, cv2.COLOR_BGR2RGB), 256), f"{cls} -> {sp} ({sc:.0%})"))
    df = pd.DataFrame(rows)
    comparable = bool(len(df)) and any(_norm(t) in model_names for t in df["true species"].unique())
    if comparable:
        df["correct"] = [_norm(a) == _norm(b) for a, b in zip(df["true species"], df["predicted"])]
    return df, comparable, gallery

def species_summary(df, comparable):
    if not len(df):
        return "No images could be read.", None
    if not comparable:
        return "Predictions made, but folder names do not match classes.", None
    acc = float(df["correct"].mean())
    per = (df.groupby("true species")["correct"].agg(["count", "sum"]).rename(columns={"count": "crops", "sum": "correct"}))
    per["accuracy (%)"] = (per["correct"] / per["crops"] * 100).round(1)
    return f"### Species top-1 accuracy: **{acc * 100:.1f}%**", per.reset_index()

def check_images(imgs, n, st, seed):
    rng = random.Random(seed)
    sample = rng.sample(imgs, min(int(n), len(imgs)))
    gallery, rows = [], []
    for f in sample:
        bgr = cv2.imread(str(f))
        if bgr is None:
            continue
        out, dets, _ = pipeline.process_image(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB), st)
        gt = label_count(f)
        sp = Counter(d["species"] for d in dets)
        rows.append({
            "file": f.name, "fish detected": len(dets), "GT fish": gt,
            "count match": None if gt is None else len(dets) == gt,
            "species": ", ".join(f"{k} x{v}" for k, v in sp.most_common()) or "-",
            "sizes": ", ".join(sorted({d["size"] for d in dets})) or "-",
            "mean det. conf.": round(float(np.mean([d["conf"] for d in dets])), 3) if dets else None,
        })
        if len(gallery) < MAX_GALLERY:
            gallery.append((_thumb(out), f"{f.name}: {len(dets)} fish"))
    return gallery, pd.DataFrame(rows)

def detect_summary(df, n_images):
    if not len(df):
        return "No images could be read."
    md = f"### Tested {len(df)} of {n_images} images\\nAverage fish detected per image: **{df['fish detected'].mean():.2f}**"
    return md

def sample_dataset(name, path, n, per_class, seed, st):
    root, imgs, vids = scan(path)
    if not imgs:
        raise ValueError(f"No images found under {root}")
    if name == SPECIES_SET:
        df, comparable, gallery = check_species(imgs, per_class, seed)
        md, per = species_summary(df, comparable)
        return {"gallery": gallery, "table": df, "summary": md, "per_class": per}
    gallery, df = check_images(imgs, n, st, seed)
    return {"gallery": gallery, "table": df, "summary": detect_summary(df, len(imgs)), "per_class": None}

def check_all(paths, n, per_class, seed, st, progress=None):
    rows = []
    for i, name in enumerate(NAMES):
        row = {"Dataset": name, "Folder found": False, "Images": None, "Videos": None, "Tested": None, "Avg fish / image": None, "GT count match (%)": None, "Species top-1 (%)": None, "Notes": ""}
        path = paths.get(name, "")
        try:
            root, imgs, vids = scan(path)
        except ValueError as e:
            row["Notes"] = str(e)
            rows.append(row)
            continue
        row.update({"Folder found": True, "Images": len(imgs), "Videos": len(vids)})
        rows.append(row)
    return pd.DataFrame(rows)
"""
with open('dataset_check.py', 'w') as f:
    f.write(dataset_check_code)

print("Written local files 'pipeline.py' and 'dataset_check.py'. Imports should work now!")

Written local files 'pipeline.py' and 'dataset_check.py'. Imports should work now!


In [37]:
import os
import shutil

# Create the directory structure expected by AquaVision
os.makedirs('/content/data/deepfish', exist_ok=True)
os.makedirs('/content/data/ozfish', exist_ok=True)
os.makedirs('/content/data/brackish', exist_ok=True)
os.makedirs('/content/data/fish4knowledge', exist_ok=True)

print("Created directories:")
print("  - /content/data/deepfish")
print("  - /content/data/ozfish")
print("  - /content/data/brackish")
print("  - /content/data/fish4knowledge")
print("\nYou can now upload or download files directly into these folders.")

Created directories:
  - /content/data/deepfish
  - /content/data/ozfish
  - /content/data/brackish
  - /content/data/fish4knowledge

You can now upload or download files directly into these folders.


### Dataset Upload & Extraction Helper
Run the cell below to upload your dataset ZIP archive. The script will automatically unpack it into the correct directory structure (`/content/data/`).

In [ ]:
from google.colab import files
import zipfile
import os

# Prompt user to upload files
uploaded = files.upload()

for filename in uploaded.keys():
    if filename.endswith('.zip'):
        print(f'Extracting {filename}...')
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            # Extracting directly into /content/data/
            zip_ref.extractall('/content/data/')
        print(f'Successfully extracted {filename} into /content/data/!')
    else:
        print(f'Uploaded non-zip file: {filename}. Please move it manually if required.')

In [38]:
"""
AquaVision - Web dashboard (Gradio)

Upload a fish image or video -> the trained models (from the training teammates) detect each
fish, track it with a unique ID (ByteTrack), classify the species, estimate relative length and
show the processed video, IDs, lengths and movement in a simple results view.

The "Datasets" tab checks the models on your own DeepFish / OZFish / Brackish / Fish4Knowledge
folders (all at once, or one at a time).

Needs two trained weight files (put them in the weights/ folder):
    weights/detector_seg.pt   YOLO11s-seg detector (single class "fish")
    weights/species_cls.pt    YOLO11s-cls species classifier (23 classes)
Without them it still runs (stock COCO detector, species = "Unknown") so the UI can be tested.

Run:  python app.py        (Colab: GRADIO_SHARE=1 python app.py)
"""

import os
from collections import Counter

import gradio as gr

import dataset_check as dc
import pipeline


# --------------------------------------------------------------------------
# helpers
# --------------------------------------------------------------------------
def make_settings(conf, iou, t_small, t_large, do_species, classify_every, max_frames,
                  px_per_cm, trail_len):
    return pipeline.Settings(
        conf=float(conf), iou=float(iou), t_small=float(t_small), t_large=float(t_large),
        do_species=bool(do_species), classify_every=int(classify_every),
        max_frames=int(max_frames), px_per_cm=float(px_per_cm or 0), trail_len=int(trail_len))


def apply_weights(det, cls):
    pipeline.set_weights(det=det.strip() or None, cls=cls.strip() or None)
    return pipeline.model_status()


def guard(fn, *a, **kw):
    """Turn plain ValueErrors from helper modules into friendly Gradio errors."""
    try:
        return fn(*a, **kw)
    except ValueError as e:
        raise gr.Error(str(e))


# --------------------------------------------------------------------------
# Image / video handlers
# --------------------------------------------------------------------------
def ui_image(image, *s):
    if image is None:
        raise gr.Error("Please upload an image.")
    st = make_settings(*s)
    out, dets, df = pipeline.process_image(image, st)
    summary = f"### Fish count: **{len(dets)}**\n"
    if dets:
        summary += "\n".join(f"- {k}: {v}" for k, v in
                             Counter(d["species"] for d in dets).most_common())
    summary += ("\n\n_Calibrated lengths are valid only for fish at the reference distance._"
                if st.px_per_cm > 0 else
                "\n\n_Size is a **relative** class (fraction of the frame diagonal); a single "
                "camera gives no metric scale._")
    return out, summary, df, pipeline.export_csv(df, "fish_detections.csv")


def ui_video(video, *s, progress=gr.Progress()):
    if video is None:
        raise gr.Error("Please upload an MP4 video.")
    st = make_settings(*s)
    res = pipeline.process_video(video, st, write_video=True, progress=progress)
    summary = pipeline.summarize_tracks(res["tracks"], res["n_frames"], res["fps"],
                                        st.px_per_cm > 0)
    return (res["out_video"], summary, res["tracks"],
            pipeline.export_csv(res["tracks"], "fish_tracks.csv"),
            pipeline.export_csv(res["frames"], "fish_per_frame.csv"))


# --------------------------------------------------------------------------
# Dataset handlers
# --------------------------------------------------------------------------
def _path_for(name, p1, p2, p3, p4):
    return [p1, p2, p3, p4][dc.NAMES.index(name)]


def ui_ds_scan(name, p1, p2, p3, p4):
    path = _path_for(name, p1, p2, p3, p4)
    info = guard(dc.scan_summary, path, True)
    md = (f"### {name}\n`{path}`  \n**{info['n_images']}** images, **{info['n_videos']}** videos")
    if info["subfolders"]:
        shown = ", ".join(f"{k} ({v})" for k, v in list(info["subfolders"].items())[:12])
        more = "" if info["n_subfolders"] <= 12 else f" ... +{info['n_subfolders'] - 12} more"
        md += f"  \nTop-level folders: {shown}{more}"
    vids = info["videos"]
    return md, gr.update(choices=vids, value=vids[0] if vids else None)


def ui_ds_sample(name, p1, p2, p3, p4, n, per_class, seed, *s, progress=gr.Progress()):
    path = _path_for(name, p1, p2, p3, p4)
    st = make_settings(*s)
    progress(0.1, desc=f"Running on {name}")
    res = guard(dc.sample_dataset, name, path, int(n), int(per_class), int(seed or 0), st)
    csv = pipeline.export_csv(res["table"], f"{name.lower()}_check.csv")
    md = res["summary"]
    per = res["per_class"]
    return res["gallery"], md, res["table"], (per if per is not None else None), csv


def ui_ds_video(name, p1, p2, p3, p4, rel, *s, progress=gr.Progress()):
    if not rel:
        raise gr.Error("Scan the dataset first and pick a video.")
    path = _path_for(name, p1, p2, p3, p4)
    full = guard(dc.resolve_video, path, rel)
    return ui_video(full, *s, progress=progress)


def ui_check_all(p1, p2, p3, p4, n, per_class, seed, *s, progress=gr.Progress()):
    st = make_settings(*s)
    paths = dict(zip(dc.NAMES, [p1, p2, p3, p4]))
    df = dc.check_all(paths, int(n), int(per_class), int(seed or 0), st, progress=progress)
    found = int(df["Folder found"].sum())
    md = f"### Checked {found} of {len(dc.NAMES)} datasets"
    if found < len(dc.NAMES):
        md += "  \nSome folders were not found - fix the paths above and run again."
    return df, md, pipeline.export_csv(df, "all_datasets_check.csv")


# --------------------------------------------------------------------------
# UI
# --------------------------------------------------------------------------
THEME = gr.themes.Soft(primary_hue="teal", secondary_hue="cyan")
DET0, CLS0 = pipeline.CFG["det"], pipeline.CFG["cls"]
DP = dc.DEFAULT_PATHS

with gr.Blocks(theme=THEME, title="AquaVision") as demo:
    gr.Markdown(
        "# AquaVision\n"
        "**Fish Movement Tracking, Length Estimation & Species Classification**  \n"
        "Upload a fish image or video: fish are detected, given unique IDs, tracked across "
        "frames, measured (relative size) and classified by species.")

    with gr.Accordion("Models & settings", open=False):
        with gr.Row():
            det_path = gr.Textbox(value=DET0, label="Detector weights (YOLO11s-seg)")
            cls_path = gr.Textbox(value=CLS0, label="Species classifier weights (YOLO11s-cls)")
            reload_btn = gr.Button("Apply / reload", scale=0)
        status = gr.Markdown(pipeline.model_status())
        reload_btn.click(apply_weights, [det_path, cls_path], status)
        with gr.Row():
            conf = gr.Slider(0.05, 0.9, value=0.25, step=0.05, label="Detection confidence")
            iou = gr.Slider(0.1, 0.9, value=0.5, step=0.05, label="NMS IoU")
        with gr.Row():
            t_small = gr.Slider(0.02, 0.3, value=0.08, step=0.01,
                                label="Small | Medium boundary (fraction of frame diagonal)")
            t_large = gr.Slider(0.1, 0.8, value=0.20, step=0.01,
                                label="Medium | Large boundary (fraction of frame diagonal)")
        with gr.Row():
            do_species = gr.Checkbox(value=True, label="Classify species")
            classify_every = gr.Slider(1, 30, value=5, step=1,
                                       label="Re-classify each track every N frames")
            max_frames = gr.Slider(30, 5000, value=900, step=30, label="Max video frames")
            trail_len = gr.Slider(5, 150, value=40, step=5,
                                  label="Movement trail length (frames)")
        px_per_cm = gr.Number(
            value=0, label="Calibration: pixels per cm (0 = off, relative size only)",
            info="Only meaningful with a scale reference; valid for fish at the same distance "
                 "as the reference.")
    S = [conf, iou, t_small, t_large, do_species, classify_every, max_frames, px_per_cm, trail_len]

    with gr.Tabs():
        # ------------------------------ Video ------------------------------
        with gr.Tab("Video"):
            with gr.Row():
                vid_in = gr.Video(label="Upload fish video (MP4)")
                vid_out = gr.Video(label="Processed video (IDs, trails, species, size)")
            vid_btn = gr.Button("Track & analyse video", variant="primary")
            vid_sum = gr.Markdown()
            vid_tbl = gr.Dataframe(label="Per-fish results (ID, species, length, movement)",
                                   interactive=False)
            with gr.Row():
                vid_csv1 = gr.File(label="Download tracks CSV")
                vid_csv2 = gr.File(label="Download per-frame CSV")
            vid_btn.click(ui_video, [vid_in] + S, [vid_out, vid_sum, vid_tbl, vid_csv1, vid_csv2])

        # ------------------------------ Image ------------------------------
        with gr.Tab("Image"):
            with gr.Row():
                img_in = gr.Image(type="numpy", label="Upload fish image",
                                  sources=["upload", "clipboard"])
                img_out = gr.Image(label="Annotated result")
            img_btn = gr.Button("Analyse image", variant="primary")
            img_sum = gr.Markdown()
            img_tbl = gr.Dataframe(label="Detections", interactive=False)
            img_csv = gr.File(label="Download CSV")
            img_btn.click(ui_image, [img_in] + S, [img_out, img_sum, img_tbl, img_csv])

        # ----------------------------- Datasets ----------------------------
        with gr.Tab("Datasets"):
            gr.Markdown(
                "Check the models on **your own dataset folders**. Detection datasets "
                "(DeepFish, OZFish, Brackish) run detect -> species -> size and compare the "
                "fish count with YOLO label files if present. **Fish4Knowledge** (one sub-folder "
                "per species) measures species accuracy on the crops.")
            with gr.Row():
                p_deep = gr.Textbox(value=DP["DeepFish"], label="DeepFish folder")
                p_oz = gr.Textbox(value=DP["OZFish"], label="OZFish folder")
                p_brack = gr.Textbox(value=DP["Brackish"], label="Brackish folder")
                p_f4k = gr.Textbox(value=DP["Fish4Knowledge"], label="Fish4Knowledge folder")
            P = [p_deep, p_oz, p_brack, p_f4k]
            with gr.Row():
                ds_n = gr.Slider(1, 100, value=12, step=1, label="Images to test per detection dataset")
                ds_pc = gr.Slider(1, 100, value=10, step=1, label="Crops per species (Fish4Knowledge)")
                ds_seed = gr.Number(value=0, precision=0, label="Random seed")

            gr.Markdown("### Check all datasets")
            all_btn = gr.Button("Check ALL datasets", variant="primary")
            all_md = gr.Markdown()
            all_tbl = gr.Dataframe(label="Summary per dataset", interactive=False)
            all_csv = gr.File(label="Download summary CSV")
            all_btn.click(ui_check_all, P + [ds_n, ds_pc, ds_seed] + S, [all_tbl, all_md, all_csv])

            gr.Markdown("### Inspect one dataset")
            with gr.Row():
                ds_pick = gr.Dropdown(dc.NAMES, value="DeepFish", label="Dataset")
                scan_btn = gr.Button("Scan folder")
                sample_btn = gr.Button("Run on random samples", variant="primary")
            ds_info = gr.Markdown()
            ds_sum = gr.Markdown()
            ds_gal = gr.Gallery(label="Results", columns=4, height=480)
            ds_tbl = gr.Dataframe(label="Per-image results", interactive=False)
            ds_per = gr.Dataframe(label="Per-species accuracy (Fish4Knowledge)", interactive=False)
            ds_csv = gr.File(label="Download CSV")
            ds_vid_pick = gr.Dropdown([], label="Video in this dataset (after scanning)")
            ds_vid_btn = gr.Button("Track selected video")
            with gr.Row():
                ds_vid_out = gr.Video(label="Processed video")
            ds_vid_sum = gr.Markdown()
            ds_vid_tbl = gr.Dataframe(label="Per-fish results", interactive=False)
            with gr.Row():
                ds_vid_c1 = gr.File(label="Tracks CSV")
                ds_vid_c2 = gr.File(label="Per-frame CSV")

            scan_btn.click(ui_ds_scan, [ds_pick] + P, [ds_info, ds_vid_pick])
            sample_btn.click(ui_ds_sample, [ds_pick] + P + [ds_n, ds_pc, ds_seed] + S,
                             [ds_gal, ds_sum, ds_tbl, ds_per, ds_csv])
            ds_vid_btn.click(ui_ds_video, [ds_pick] + P + [ds_vid_pick] + S,
                             [ds_vid_out, ds_vid_sum, ds_vid_tbl, ds_vid_c1, ds_vid_c2])

        # ------------------------------- About -----------------------------
        with gr.Tab("About"):
            gr.Markdown(
                "**Pipeline:** YOLO11s-seg detects fish and masks -> YOLO11s-cls labels each "
                "crop with one of 23 species -> PCA on the mask gives length (relative "
                "size class; cm only with a calibration scale) -> ByteTrack gives persistent IDs "
                "so each fish is counted once.  \n"
                "**Movement:** path length, net displacement, speed and heading per fish, plus "
                "trails drawn on the video.  \n"
                "**Team:** Mihiran Chakraborty, Aditya Khuttan, Mudaliar Krishnan, Aditya Sahu - "
                "AI Lab, Ahmedabad University.")

if __name__ == "__main__":
    demo.queue().launch(share=os.getenv("GRADIO_SHARE") == "1")

/tmp/ipykernel_637/1630953423.py:139: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=THEME, title="AquaVision") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

In [34]:
"""
Dataset checking for the AquaVision dashboard.

Lets you point the dashboard at DeepFish, OZFish, Brackish and Fish4Knowledge folders and
see how the trained models behave on each:

  * detection datasets (DeepFish / OZFish / Brackish): run detect -> species -> size on random
    samples; if YOLO label files exist next to the images, compare detected vs ground-truth
    fish count.
  * Fish4Knowledge (species crops, one sub-folder per species): run the species classifier on
    random crops and report top-1 accuracy (overall + per class).

No gradio import here, so it can be tested on its own. Errors are raised as ValueError.
"""

import random
import re
from collections import Counter
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

import pipeline

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
VID_EXT = {".mp4", ".avi", ".mov", ".mkv"}
NAMES = ["DeepFish", "OZFish", "Brackish", "Fish4Knowledge"]
SPECIES_SET = "Fish4Knowledge"
DEFAULT_PATHS = {
    "DeepFish": "/content/data/deepfish",
    "OZFish": "/content/data/ozfish",
    "Brackish": "/content/data/brackish",
    "Fish4Knowledge": "/content/data/fish4knowledge",
}
MAX_GALLERY = 24
_cache = {}


# --------------------------------------------------------------------------
# scanning
# --------------------------------------------------------------------------
def scan(path, refresh=False):
    p = Path(str(path).strip())
    if not p.exists() or not p.is_dir():
        raise ValueError(f"Folder not found: {path}")
    key = str(p.resolve())
    if refresh or key not in _cache:
        imgs, vids = [], []
        for f in sorted(p.rglob("*")):
            ext = f.suffix.lower()
            if ext in IMG_EXT:
                imgs.append(f)
            elif ext in VID_EXT:
                vids.append(f)
        _cache[key] = (p, imgs, vids)
    return _cache[key]


def scan_summary(path, refresh=False):
    root, imgs, vids = scan(path, refresh)
    sub = Counter()
    for f in imgs:
        rel = f.relative_to(root)
        sub[rel.parts[0] if len(rel.parts) > 1 else "(top level)"] += 1
    return {
        "n_images": len(imgs), "n_videos": len(vids),
        "videos": [str(v.relative_to(root)) for v in vids[:500]],
        "subfolders": dict(sub.most_common(30)),
        "n_subfolders": len(sub),
    }


def resolve_video(path, rel):
    root, _, _ = scan(path)
    f = (root / rel)
    if not f.exists():
        raise ValueError(f"Video not found: {f}")
    return str(f)


def label_count(img_path):
    """Number of YOLO label lines for an image (labels/ mirror of images/, or .txt beside it)."""
    parts = list(Path(img_path).parts)
    lp = None
    for i in range(len(parts) - 1, -1, -1):
        if parts[i] == "images":
            parts[i] = "labels"
            lp = Path(*parts).with_suffix(".txt")
            break
    if lp is None:
        lp = Path(img_path).with_suffix(".txt")
    if not lp.exists():
        return None
    return sum(1 for ln in lp.read_text().splitlines() if ln.strip())


def _thumb(rgb, side=640):
    h, w = rgb.shape[:2]
    s = side / max(h, w)
    return cv2.resize(rgb, (int(w * s), int(h * s))) if s < 1 else rgb


def _norm(s):
    return re.sub(r"[\W_]+", "", str(s).lower())


# --------------------------------------------------------------------------
# Fish4Knowledge: species accuracy on crops
# --------------------------------------------------------------------------
def check_species(imgs, per_class, seed):
    clf = pipeline.get_classifier()
    if clf is None:
        raise ValueError("Species classifier weights not found - set them in 'Models & settings'.")
    by = {}
    for f in imgs:
        by.setdefault(f.parent.name, []).append(f)
    if len(by) < 2:
        raise ValueError("Expected one sub-folder per species (found fewer than 2 folders).")
    rng = random.Random(seed)
    model_names = {_norm(v) for v in clf.names.values()}
    rows, gallery = [], []
    for cls, files in sorted(by.items()):
        for f in rng.sample(files, min(int(per_class), len(files))):
            im = cv2.imread(str(f))
            if im is None:
                continue
            sp, sc = pipeline.classify_crop(im)
            rows.append({"file": f.name, "true species": cls, "predicted": sp,
                         "confidence": round(sc, 3)})
            if len(gallery) < MAX_GALLERY:
                gallery.append((_thumb(cv2.cvtColor(im, cv2.COLOR_BGR2RGB), 256),
                                f"{cls} -> {sp} ({sc:.0%})"))
    df = pd.DataFrame(rows)
    comparable = bool(len(df)) and any(_norm(t) in model_names for t in df["true species"].unique())
    if comparable:
        df["correct"] = [_norm(a) == _norm(b) for a, b in zip(df["true species"], df["predicted"])]
    return df, comparable, gallery


def species_summary(df, comparable):
    if not len(df):
        return "No images could be read.", None
    if not comparable:
        return ("Predictions made, but the folder names do not match the classifier's class "
                "names, so accuracy cannot be computed. Rename the folders to the class names "
                "used in training."), None
    acc = float(df["correct"].mean())
    per = (df.groupby("true species")["correct"].agg(["count", "sum"])
           .rename(columns={"count": "crops", "sum": "correct"}))
    per["accuracy (%)"] = (per["correct"] / per["crops"] * 100).round(1)
    macro = float(per["accuracy (%)"].mean())
    md = (f"### Species top-1 accuracy: **{acc * 100:.1f}%**  \n"
          f"{len(df)} crops from {df['true species'].nunique()} species; "
          f"mean per-class accuracy {macro:.1f}%.")
    return md, per.reset_index()


# --------------------------------------------------------------------------
# Detection datasets: run the full image pipeline on samples
# --------------------------------------------------------------------------
def check_images(imgs, n, st, seed):
    rng = random.Random(seed)
    sample = rng.sample(imgs, min(int(n), len(imgs)))
    gallery, rows = [], []
    for f in sample:
        bgr = cv2.imread(str(f))
        if bgr is None:
            continue
        out, dets, _ = pipeline.process_image(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB), st)
        gt = label_count(f)
        sp = Counter(d["species"] for d in dets)
        rows.append({
            "file": f.name, "fish detected": len(dets), "GT fish": gt,
            "count match": None if gt is None else len(dets) == gt,
            "species": ", ".join(f"{k} x{v}" for k, v in sp.most_common()) or "-",
            "sizes": ", ".join(sorted({d["size"] for d in dets})) or "-",
            "mean det. conf.": round(float(np.mean([d["conf"] for d in dets])), 3) if dets else None,
        })
        if len(gallery) < MAX_GALLERY:
            gallery.append((_thumb(out), f"{f.name}: {len(dets)} fish"))
    return gallery, pd.DataFrame(rows)


def detect_summary(df, n_images):
    if not len(df):
        return "No images could be read."
    md = (f"### Tested {len(df)} of {n_images} images  \n"
          f"Average fish detected per image: **{df['fish detected'].mean():.2f}**")
    g = df.dropna(subset=["GT fish"])
    if len(g):
        match = float(g["count match"].astype(bool).mean() * 100)
        mae = float((g["fish detected"] - g["GT fish"].astype(float)).abs().mean())
        md += (f"  \nGround-truth labels found for {len(g)} images: count matches on "
               f"**{match:.0f}%**, mean absolute count error **{mae:.2f}**.")
    else:
        md += "  \nNo YOLO label files found next to these images, so no GT comparison."
    return md


# --------------------------------------------------------------------------
# One dataset / all datasets
# --------------------------------------------------------------------------
def sample_dataset(name, path, n, per_class, seed, st):
    """-> dict(gallery, table, summary, per_class_table)"""
    root, imgs, vids = scan(path)
    if not imgs:
        raise ValueError(f"No images found under {root}")
    if name == SPECIES_SET:
        df, comparable, gallery = check_species(imgs, per_class, seed)
        md, per = species_summary(df, comparable)
        return {"gallery": gallery, "table": df, "summary": md, "per_class": per}
    gallery, df = check_images(imgs, n, st, seed)
    return {"gallery": gallery, "table": df, "summary": detect_summary(df, len(imgs)),
            "per_class": None}


def check_all(paths, n, per_class, seed, st, progress=None):
    """paths: {dataset name: folder}. Returns one summary row per dataset."""
    rows = []
    for i, name in enumerate(NAMES):
        if progress is not None:
            progress(i / len(NAMES), desc=f"Checking {name}")
        row = {"Dataset": name, "Folder found": False, "Images": None, "Videos": None,
               "Tested": None, "Avg fish / image": None, "GT count match (%)": None,
               "Species top-1 (%)": None, "Notes": ""}
        path = paths.get(name, "")
        try:
            root, imgs, vids = scan(path)
        except ValueError as e:
            row["Notes"] = str(e)
            rows.append(row)
            continue
        row.update({"Folder found": True, "Images": len(imgs), "Videos": len(vids)})
        if not imgs:
            row["Notes"] = "no images found"
            rows.append(row)
            continue
        try:
            if name == SPECIES_SET:
                df, comparable, _ = check_species(imgs, per_class, seed)
                row["Tested"] = len(df)
                if comparable:
                    row["Species top-1 (%)"] = round(float(df["correct"].mean()) * 100, 1)
                else:
                    row["Notes"] = "folder names do not match classifier classes"
            else:
                _, df = check_images(imgs, n, st, seed)
                row["Tested"] = len(df)
                if len(df):
                    row["Avg fish / image"] = round(float(df["fish detected"].mean()), 2)
                    g = df.dropna(subset=["GT fish"])
                    if len(g):
                        row["GT count match (%)"] = round(float(g["count match"].astype(bool).mean()) * 100, 1)
                    else:
                        row["Notes"] = "no YOLO labels found (no GT comparison)"
        except ValueError as e:
            row["Notes"] = str(e)
        rows.append(row)
    if progress is not None:
        progress(1.0, desc="Done")
    return pd.DataFrame(rows)

In [35]:
"""
AquaVision inference pipeline.

    frame -> YOLO11-seg (boxes + masks) -> YOLO11-cls (23 species, on crops)
                                        -> PCA on mask  (relative size-class, optional cm)
                                        -> ByteTrack    (IDs, de-duplicated count, movement)
"""

import os
import shutil
import subprocess
import tempfile
from collections import Counter, defaultdict
from dataclasses import dataclass

import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO

# --------------------------------------------------------------------------
# Config / models
# --------------------------------------------------------------------------
CFG = {
    "det": os.getenv("DET_WEIGHTS", "weights/detector_seg.pt"),
    "cls": os.getenv("CLS_WEIGHTS", "weights/species_cls.pt"),
}
DET_IMGSZ = 640
CLS_IMGSZ = 224
MAX_VOTES_PER_TRACK = 12
SIZE_LABELS = ["Small", "Medium", "Large"]

_models = {"det": None, "cls": None}


@dataclass
class Settings:
    conf: float = 0.25
    iou: float = 0.5
    t_small: float = 0.08          # Small/Medium boundary (fraction of frame diagonal)
    t_large: float = 0.20          # Medium/Large boundary
    do_species: bool = True
    classify_every: int = 5        # per-track species re-check stride (frames)
    max_frames: int = 900
    px_per_cm: float = 0.0         # >0 enables calibrated length (needs a scale reference)
    trail_len: int = 40            # movement trail length (frames)
    min_track_frames: int = 3      # drop flicker tracks shorter than this


def set_weights(det=None, cls=None):
    if det:
        CFG["det"] = det
    if cls:
        CFG["cls"] = cls
    reload_models()


def reload_models():
    _models["det"] = None
    _models["cls"] = None


def _load(path, fallback=None):
    if path and os.path.exists(path):
        return YOLO(path)
    if fallback:
        print(f"[warn] {path} not found, using {fallback}")
        return YOLO(fallback)
    print(f"[warn] {path} not found")
    return None


def get_detector(fresh=False, weights=None):
    """fresh=True -> new instance (resets ByteTrack state between videos)."""
    if weights:
        return YOLO(weights)
    if fresh:
        return _load(CFG["det"], "yolo11s-seg.pt")
    if _models["det"] is None:
        _models["det"] = _load(CFG["det"], "yolo11s-seg.pt")
    return _models["det"]


def get_classifier():
    if _models["cls"] is None:
        _models["cls"] = _load(CFG["cls"])
    return _models["cls"]


def model_status():
    d = "found" if os.path.exists(CFG["det"]) else "MISSING (using stock COCO yolo11s-seg)"
    c = "found" if os.path.exists(CFG["cls"]) else "MISSING (species = Unknown)"
    return f"Detector `{CFG['det']}`: {d}  \nClassifier `{CFG['cls']}`: {c}"


# --------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------
def classify_crop(crop_bgr):
    clf = get_classifier()
    if clf is None or crop_bgr is None or crop_bgr.size == 0:
        return "Unknown", 0.0
    res = clf.predict(crop_bgr, imgsz=CLS_IMGSZ, verbose=False)[0]
    idx = int(res.probs.top1)
    return clf.names[idx], float(res.probs.top1conf)


def mask_pixels(poly, shape):
    """Rasterise polygon (N,2) inside its bbox -> (M,2) array of (x, y) pixels."""
    h, w = shape[:2]
    x0, y0 = np.floor(poly.min(0)).astype(int)
    x1, y1 = np.ceil(poly.max(0)).astype(int)
    x0, y0 = max(x0, 0), max(y0, 0)
    x1, y1 = min(x1, w - 1), min(y1, h - 1)
    if x1 <= x0 or y1 <= y0:
        return None
    canvas = np.zeros((y1 - y0 + 1, x1 - x0 + 1), np.uint8)
    cv2.fillPoly(canvas, [(poly - np.array([x0, y0])).astype(np.int32)], 1)
    ys, xs = np.nonzero(canvas)
    if len(xs) < 5:
        return None
    return np.stack([xs + x0, ys + y0], axis=1).astype(np.float32)


def pca_length(points):
    """Extent along the first principal axis (via SVD) of the mask pixels."""
    pts = points - points.mean(0)
    _, _, vt = np.linalg.svd(pts, full_matrices=False)
    proj = pts @ vt[0]
    return float(proj.max() - proj.min() + 1.0)


def size_class(rel_len, t_small, t_large):
    if rel_len < t_small:
        return SIZE_LABELS[0]
    if rel_len < t_large:
        return SIZE_LABELS[1]
    return SIZE_LABELS[2]


def colour_for(key):
    rng = np.random.RandomState(abs(hash(str(key))) % (2**31))
    return tuple(int(c) for c in rng.randint(60, 255, 3))


def draw_detection(img, det, alpha=0.35):
    colour = colour_for(det["id"] if det["id"] is not None else det["species"])
    poly = det.get("poly")
    if poly is not None and len(poly) >= 3:
        overlay = img.copy()
        cv2.fillPoly(overlay, [poly.astype(np.int32)], colour)
        cv2.addWeighted(overlay, alpha, img, 1 - alpha, 0, img)
        cv2.polylines(img, [poly.astype(np.int32)], True, colour, 2)
    x1, y1, x2, y2 = [int(v) for v in det["box"]]
    cv2.rectangle(img, (x1, y1), (x2, y2), colour, 2)

    tag = f"ID {det['id']} | " if det["id"] is not None else ""
    label = f"{tag}{det['species']} {det['sconf']:.0%} | {det['size']}"
    if det.get("length_cm") is not None:
        label += f" | {det['length_cm']:.1f} cm"
    (tw, th), bl = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
    ty = max(y1, th + 6)
    cv2.rectangle(img, (x1, ty - th - 6), (x1 + tw + 4, ty + bl - 4), colour, -1)
    cv2.putText(img, label, (x1 + 2, ty - 4), cv2.FONT_HERSHEY_SIMPLEX,
                0.5, (0, 0, 0), 1, cv2.LINE_AA)


def extract_detections(result, frame, st, votes=None, frame_idx=0, scale=1.0):
    """One Ultralytics result -> list of detection dicts."""
    dets = []
    if result.boxes is None or len(result.boxes) == 0:
        return dets

    h, w = frame.shape[:2]
    diag = float(np.hypot(h, w))
    boxes = result.boxes.xyxy.cpu().numpy()
    confs = result.boxes.conf.cpu().numpy()
    ids = (result.boxes.id.cpu().numpy().astype(int)
           if result.boxes.id is not None else [None] * len(boxes))
    polys = result.masks.xy if result.masks is not None else [None] * len(boxes)

    for box, conf, tid, poly in zip(boxes, confs, ids, polys):
        x1, y1 = max(int(box[0]), 0), max(int(box[1]), 0)
        x2, y2 = min(int(box[2]), w), min(int(box[3]), h)
        if x2 - x1 < 4 or y2 - y1 < 4:
            continue

        # ---- relative (and optional calibrated) length ----
        pts = None
        if poly is not None and len(poly) >= 3:
            pts = mask_pixels(np.asarray(poly), frame.shape)
        if pts is None:
            pts = np.array([[x1, y1], [x2, y1], [x2, y2], [x1, y2]], np.float32)
        length_px = pca_length(pts)
        rel = length_px / diag
        length_cm = None
        if st.px_per_cm and st.px_per_cm > 0:
            length_cm = (length_px / scale) / st.px_per_cm

        # ---- species (per-track vote when tracked) ----
        species, sconf = "Unknown", 0.0
        if st.do_species:
            crop = frame[y1:y2, x1:x2]
            if tid is None or votes is None:
                species, sconf = classify_crop(crop)
            else:
                v = votes[tid]
                n = len(v["confs"])
                if n == 0 or (n < MAX_VOTES_PER_TRACK and frame_idx % max(st.classify_every, 1) == 0):
                    sp, sc = classify_crop(crop)
                    v["votes"][sp] += sc
                    v["confs"].append(sc)
                species = v["votes"].most_common(1)[0][0]
                sconf = float(np.mean(v["confs"]))

        dets.append(dict(
            id=tid, box=(x1, y1, x2, y2), poly=None if poly is None else np.asarray(poly),
            conf=float(conf), species=species, sconf=sconf, length_px=length_px,
            rel=rel, length_cm=length_cm, size=size_class(rel, st.t_small, st.t_large),
            centre=((x1 + x2) / 2.0, (y1 + y2) / 2.0),
        ))
    return dets


def export_csv(df, name):
    path = os.path.join(tempfile.mkdtemp(), name)
    df.to_csv(path, index=False)
    return path


# --------------------------------------------------------------------------
# Image
# --------------------------------------------------------------------------
IMG_COLS = ["Fish", "Species", "Species conf.", "Size class", "Rel. length (% diag)",
            "Length (cm)", "Det. conf."]


def process_image(image_rgb, st, det_weights=None):
    det_model = get_detector(weights=det_weights)
    bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)
    res = det_model.predict(bgr, imgsz=DET_IMGSZ, conf=st.conf, iou=st.iou, verbose=False)[0]
    dets = extract_detections(res, bgr, st)
    out = bgr.copy()
    for d in dets:
        draw_detection(out, d)
    rows = [{
        "Fish": i + 1, "Species": d["species"], "Species conf.": round(d["sconf"], 3),
        "Size class": d["size"], "Rel. length (% diag)": round(d["rel"] * 100, 1),
        "Length (cm)": None if d["length_cm"] is None else round(d["length_cm"], 1),
        "Det. conf.": round(d["conf"], 3),
    } for i, d in enumerate(dets)]
    df = pd.DataFrame(rows, columns=IMG_COLS)
    return cv2.cvtColor(out, cv2.COLOR_BGR2RGB), dets, df


# --------------------------------------------------------------------------
# Video (ByteTrack, trails, movement)
# --------------------------------------------------------------------------
def _reencode_h264(src):
    if not shutil.which("ffmpeg"):
        return src
    dst = src.replace(".mp4", "_h264.mp4")
    try:
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", src,
                        "-vcodec", "libx264", "-pix_fmt", "yuv420p", dst], check=True)
        return dst
    except Exception:
        return src


def _movement(path, fps, first, last, diag):
    """Movement stats of one track (all relative to frame diagonal)."""
    p = np.asarray(path, np.float32)
    if len(p) < 2:
        return 0.0, 0.0, 0.0, 0.0
    steps = np.linalg.norm(np.diff(p, axis=0), axis=1)
    path_len = float(steps.sum())
    net = p[-1] - p[0]
    disp = float(np.linalg.norm(net))
    dur = max((last - first) / fps, 1e-6)
    heading = float(np.degrees(np.arctan2(-net[1], net[0])))   # 0=right, 90=up
    return path_len / diag * 100, disp / diag * 100, path_len / diag * 100 / dur, heading


def process_video(video_path, st, write_video=True, max_width=1280, progress=None,
                  det_weights=None):
    """Track fish through a clip.

    Returns dict(out_video, tracks, frames, frame_records, fps, n_frames, count)
    frame_records: list of (frame_idx0, track_id, x1, y1, x2, y2) in ORIGINAL pixels.
    """
    det_model = get_detector(fresh=True, weights=det_weights)
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError("Could not open the video.")
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 0
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    scale = min(1.0, max_width / W) if (write_video and max_width) else 1.0
    outW, outH = max(int(W * scale) // 2 * 2, 2), max(int(H * scale) // 2 * 2, 2)
    diag = float(np.hypot(outW, outH))

    writer, raw_path = None, None
    if write_video:
        raw_path = os.path.join(tempfile.mkdtemp(), "annotated.mp4")
        writer = cv2.VideoWriter(raw_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (outW, outH))

    votes = defaultdict(lambda: {"votes": Counter(), "confs": []})
    info = defaultdict(lambda: {"rel": [], "px": [], "det": [], "first": None,
                                "last": None, "n": 0, "path": []})
    records, frame_rows = [], []

    limit = int(st.max_frames) if st.max_frames else (total or 10**9)
    n_total = min(total, limit) if total else 0
    idx = 0
    while idx < limit:
        ok, frame = cap.read()
        if not ok:
            break
        if scale < 1.0:
            frame = cv2.resize(frame, (outW, outH))

        res = det_model.track(frame, persist=True, tracker="bytetrack.yaml",
                              imgsz=DET_IMGSZ, conf=st.conf, iou=st.iou, verbose=False)[0]
        dets = extract_detections(res, frame, st, votes=votes, frame_idx=idx, scale=scale)

        for d in dets:
            tid = d["id"]
            if tid is None:
                continue
            ti = info[tid]
            ti["rel"].append(d["rel"])
            ti["px"].append(d["length_px"] / scale)
            ti["det"].append(d["conf"])
            ti["first"] = idx if ti["first"] is None else ti["first"]
            ti["last"] = idx
            ti["n"] += 1
            ti["path"].append(d["centre"])
            d["size"] = size_class(float(np.median(ti["rel"])), st.t_small, st.t_large)
            x1, y1, x2, y2 = d["box"]
            records.append((idx, tid, x1 / scale, y1 / scale, x2 / scale, y2 / scale))
            frame_rows.append({
                "frame": idx + 1, "time_s": round(idx / fps, 3), "track_id": tid,
                "species": d["species"], "species_conf": round(d["sconf"], 3),
                "size_class": d["size"], "rel_length_pct_diag": round(d["rel"] * 100, 2),
                "length_cm": None if d["length_cm"] is None else round(d["length_cm"], 2),
                "x1": round(x1 / scale, 1), "y1": round(y1 / scale, 1),
                "x2": round(x2 / scale, 1), "y2": round(y2 / scale, 1),
            })

        if write_video:
            for d in dets:                       # movement trails
                if d["id"] is None:
                    continue
                path = info[d["id"]]["path"][-max(int(st.trail_len), 2):]
                if len(path) >= 2:
                    cv2.polylines(frame, [np.array(path, np.int32)], False,
                                  colour_for(d["id"]), 2, cv2.LINE_AA)
            for d in dets:
                draw_detection(frame, d)
            seen = sum(1 for t in info.values() if t["n"] >= st.min_track_frames)
            cv2.rectangle(frame, (0, 0), (270, 30), (0, 0, 0), -1)
            cv2.putText(frame, f"Unique fish so far: {seen}", (6, 21),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1, cv2.LINE_AA)
            writer.write(frame)

        idx += 1
        if progress is not None and n_total:
            progress(idx / n_total, desc=f"Frame {idx}/{n_total}")

    cap.release()
    if writer is not None:
        writer.release()
    out_video = _reencode_h264(raw_path) if write_video else None

    # ---- per-track table ----
    rows = []
    for tid, ti in sorted(info.items()):
        if ti["n"] < st.min_track_frames:
            continue
        v = votes[tid]
        species = v["votes"].most_common(1)[0][0] if v["votes"] else "Unknown"
        sconf = float(np.mean(v["confs"])) if v["confs"] else 0.0
        rel = float(np.median(ti["rel"]))
        path_len, disp, speed, heading = _movement(ti["path"], fps, ti["first"], ti["last"], diag)
        row = {
            "Track ID": tid, "Species": species, "Species conf.": round(sconf, 3),
            "Size class": size_class(rel, st.t_small, st.t_large),
            "Rel. length (% diag)": round(rel * 100, 1),
            "Length (cm)": (round(float(np.median(ti["px"])) / st.px_per_cm, 1)
                            if st.px_per_cm and st.px_per_cm > 0 else None),
            "Frames seen": ti["n"],
            "First (s)": round(ti["first"] / fps, 1), "Last (s)": round(ti["last"] / fps, 1),
            "Path (% diag)": round(path_len, 1), "Net disp. (% diag)": round(disp, 1),
            "Speed (% diag/s)": round(speed, 1), "Heading (deg)": round(heading, 0),
        }
        rows.append(row)
    return dict(out_video=out_video, tracks=pd.DataFrame(rows), frames=pd.DataFrame(frame_rows),
                frame_records=records, fps=fps, n_frames=idx, count=len(rows))


def summarize_tracks(tracks, n_frames, fps, calibrated):
    s = f"### Unique fish (de-duplicated by ByteTrack): **{len(tracks)}**\n"
    s += f"_Processed {n_frames} frames at {fps:.1f} FPS._\n\n"
    if len(tracks):
        sp = Counter(tracks["Species"])
        s += "\n".join(f"- {k}: {v}" for k, v in sp.most_common())
        sz = Counter(tracks["Size class"])
        s += "\n\n**Size classes:** " + ", ".join(f"{k}: {v}" for k, v in sz.items())
        s += (f"\n\n**Mean speed:** {tracks['Speed (% diag/s)'].mean():.1f} % of frame diagonal / s")
    s += ("\n\n_Calibrated lengths use your px/cm scale and are valid only for fish at about the "
          "same distance as the reference._" if calibrated else
          "\n\n_Size is a **relative** class (fraction of frame diagonal); a single camera has no "
          "metric scale. Enter px/cm in Settings if you have a scale reference._")
    return s